# U-ADMM Parameter Grid Search
This notebook runs experiments across 5 random seeds and various parameter combinations, saving all intermediate data for analysis.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('.'))
%load_ext autoreload
%autoreload 2
import time
import numpy as np
import pandas as pd
import pickle
import itertools
from models.ranking import generate_ranking_data
from algorithms.admm import run_u_admm
from utils.eval_utils import calculate_metrics, evaluate_ranking_accuracy, evaluate_correlation


In [ ]:
# 1. Define Parameter Grid
seeds = [42, 65, 123, 999, 2026]
alpha_list = [0.95, 0.6]
W_inner_list = [5, 10]
lam_t = 0.15
rho = 3.3
T = 20

# Fixed data generation params
m, n, p_prime, p, pc = 100, 100, 5, 20, 0.3
noise_type = 'cauchy'

combinations = list(itertools.product(alpha_list, W_inner_list))
print(f'Total seeds: {len(seeds)}')
print(f'Combinations per seed: {len(combinations)}')
print(f'Total runs: {len(seeds) * len(combinations)}')


In [ ]:
# 2. Run Experiments
os.makedirs('exp_results', exist_ok=True)
all_results = []

for seed in seeds:
    print(f'\n========== Seed {seed} ==========')
    np.random.seed(seed)
    d_rank = generate_ranking_data(
        m=m, n=n, p_prime=p_prime, p=p, pc=pc, 
        noise_type=noise_type, rng_seed=seed
    )
    theta_true = d_rank['theta_true']
    
    for alpha, W_inner in combinations:
        run_name = f'seed{seed}_alpha{alpha}_W{W_inner}'
        print(f'\n--- Running: {run_name} ---')
        
        t0 = time.time()
        theta_u_r, theta_n_r, hist_r = run_u_admm(
            d_rank, T=T, W_inner=W_inner, 
            rho=rho, lam_t=lam_t, alpha=alpha, verbose=False,
            lambda_candidates=[0.1, 0.05, 0.01, 0.005, 0.001],
            ic_type='bic' # 可以改为 'aic'
        )
        t_cost = time.time() - t0
        
        theta_uadmm = theta_u_r[0]
        metrics = calculate_metrics(theta_true, theta_uadmm)
        corr_metrics = evaluate_correlation(d_rank['X'], theta_true, theta_uadmm)
        
        # Save intermediate data to pickle
        pkl_path = f'exp_results/{run_name}.pkl'
        with open(pkl_path, 'wb') as f:
            pickle.dump({
                'seed': seed,
                'theta_true': theta_true,
                'theta_uadmm': theta_uadmm,
                'history': hist_r
            }, f)
        
        # Record summary metrics
        all_results.append({
            'Seed': seed,
            'Alpha': alpha,
            'W_inner': W_inner,
            'RMSE': metrics['RMSE'],
            'MAE': metrics['MAE'],
            'Selection_Acc': metrics['Selection_Acc'],
            'F1_Score': metrics['F1_Score'],
            'Pearson_Corr': corr_metrics['Pearson_Corr'],
            'Kendall_Corr': corr_metrics['Kendall_Corr'],
            'Pairwise_Correlation': (corr_metrics['Kendall_Corr'] + 1) / 2,
            'Time(s)': t_cost,
            'Pickle_Path': pkl_path
        })
        print(f"RMSE: {metrics['RMSE']:.4f}, Time: {t_cost:.1f}s")


In [ ]:
# 3. Summarize and Save Results
df_results = pd.DataFrame(all_results)
df_results.to_csv('exp_results/grid_search_summary.csv', index=False)
try:
    df_results.to_excel('exp_results/grid_search_summary.xlsx', index=False)
except Exception as e:
    print(f'Could not save to excel: {e}')
print('\nAll experiments finished! Summary saved to exp_results/grid_search_summary.csv')
display(df_results.groupby(['Adaptive_Rho', 'Alpha', 'W_inner'])[['RMSE', 'Time(s)']].mean())
